In [1]:
import pandas as pd

In [20]:
df = pd.read_csv('../data/Final Data/Chl-a/Chl-a-7-day.csv')

In [21]:
df.head()

,Chl-a,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A
0,28.47,0.0147,0.0125,0.0501,0.0725,0.0376,0.0433,0.0251,0.0241,0.0221,0.0206
1,16.77,0.0141,0.0111,0.0511,0.0714,0.0370,0.0363,0.0228,0.0233,0.0197,0.0182
2,14.30,0.0211,0.0192,0.0589,0.0783,0.0433,0.0444,0.0306,0.0312,0.0272,0.0272
3,4.64,0.0152,0.0127,0.0473,0.0532,0.0253,0.0241,0.0192,0.0211,0.0189,0.0176
4,20.48,0.0055,0.0037,0.0536,0.0713,0.0346,0.0344,0.0178,0.0160,0.0156,0.0130


In [22]:
def classify_chla(value):
    if value < 2:  # Very low concentrations, minimal algal growth
        return 0  # no algal growth
    elif 2 <= value <= 10:  # Normal or baseline levels
        return 1  # normal algal growth
    else:  # Concentrations above 10 µg/L often indicate higher algal activity
        return 2  # high algal growth

df['chl-a level'] = df['Chl-a'].apply(classify_chla)


In [23]:
df

,Chl-a,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,chl-a level
0,28.47000,0.01470,0.01250,0.05010,0.07250,0.03760,0.04330,0.02510,0.02410,0.02210,0.02060,2
1,16.77000,0.01410,0.01110,0.05110,0.07140,0.03700,0.03630,0.02280,0.02330,0.01970,0.01820,2
2,14.30000,0.02110,0.01920,0.05890,0.07830,0.04330,0.04440,0.03060,0.03120,0.02720,0.02720,2
3,4.64000,0.01520,0.01270,0.04730,0.05320,0.02530,0.02410,0.01920,0.02110,0.01890,0.01760,1
4,20.48000,0.00550,0.00370,0.05360,0.07130,0.03460,0.03440,0.01780,0.01600,0.01560,0.01300,2
...,...,...,...,...,...,...,...,...,...,...,...,...
3815,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565,0
3816,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565,0
3817,1.00661,0.13390,0.06985,0.01245,0.03465,0.02350,0.05375,0.22540,0.31595,0.31955,0.32565,0
3818,0.27000,0.00535,0.00355,0.04060,0.04695,0.01710,0.01650,0.01765,0.01765,0.01370,0.01330,0


In [24]:
df['chl-a level'].value_counts()

chl-a level
1    2284
0    1457
2      79
Name: count, dtype: int64

In [25]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=[0, 1, 2],
    y=df['chl-a level']
)

class_weights_dict = {i : class_weights[i] for i in range(3)}


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [27]:
X = df.drop(['Chl-a','chl-a level'], axis=1)
y = df['chl-a level']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# 👉 Step 4: Compute class weights
weights = class_weight.compute_class_weight(class_weight='balanced', classes=[0,1,2], y=y_train)
class_weights_dict = {i: weights[i] for i in range(3)}

In [28]:
clf = RandomForestClassifier(random_state=42, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight={0: 0.8737419945105215,
                                     1: 0.5575014594279043,
                                     2: 16.1864406779661},
                       random_state=42)

In [29]:
y_pred = clf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90       364
           1       0.95      0.91      0.93       571
           2       0.85      0.85      0.85        20

    accuracy                           0.91       955
   macro avg       0.89      0.89      0.89       955
weighted avg       0.92      0.91      0.91       955

Confusion Matrix:
 [[337  27   0]
 [ 49 519   3]
 [  0   3  17]]


In [35]:
# For turbidity
df = pd.read_csv('../data/Final Data/Turbidity/turb_train_data_7_days.csv')

In [37]:
df['TURB'].describe()

count    1438.000000
mean       16.102670
std        14.969893
min         0.100000
25%         8.100000
50%        13.650000
75%        19.775000
max       157.500000
Name: TURB, dtype: float64

In [40]:
df

,TURB,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,turbidity_class
0,18.9,0.00320,0.00460,0.03140,0.04210,0.0446,0.03990,0.01760,0.01540,0.01100,0.00760,2
1,23.4,0.00320,0.00460,0.03140,0.04210,0.0446,0.03990,0.01760,0.01540,0.01100,0.00760,2
2,19.5,0.00320,0.00460,0.03140,0.04210,0.0446,0.03990,0.01760,0.01540,0.01100,0.00760,2
3,19.3,0.00320,0.00460,0.03140,0.04210,0.0446,0.03990,0.01760,0.01540,0.01100,0.00760,2
4,18.8,0.00320,0.00460,0.03140,0.04210,0.0446,0.03990,0.01760,0.01540,0.01100,0.00760,2
...,...,...,...,...,...,...,...,...,...,...,...,...
1433,157.5,0.00905,0.00445,0.06700,0.10560,0.1076,0.12295,0.07900,0.07990,0.06285,0.05085,2
1434,91.0,0.00660,0.00605,0.08085,0.12070,0.1125,0.11595,0.05495,0.05055,0.04015,0.02800,2
1435,113.0,0.00985,0.00875,0.06585,0.09890,0.0982,0.10845,0.06610,0.06535,0.05205,0.04060,2
1436,111.0,0.05165,0.03815,0.05980,0.09225,0.0832,0.11740,0.09825,0.11340,0.09150,0.09380,2


In [38]:
def classify_turbidity(value):
    if value < 1:
        return 0  # Low
    elif value <= 10:
        return 1  # Normal
    else:
        return 2  # High

df['turbidity_class'] = df['TURB'].apply(classify_turbidity)


In [39]:
df['turbidity_class'].value_counts()


turbidity_class
2    944
1    488
0      6
Name: count, dtype: int64

In [41]:
X = df.drop(['TURB','turbidity_class'], axis=1)
y = df['turbidity_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# 👉 Step 4: Compute class weights
weights = class_weight.compute_class_weight(class_weight='balanced', classes=[0,1,2], y=y_train)
class_weights_dict = {i: weights[i] for i in range(3)}

In [42]:
clf = RandomForestClassifier(random_state=42, class_weight=class_weights_dict)
clf.fit(X_train, y_train)

RandomForestClassifier(class_weight={0: 89.83333333333333,
                                     1: 0.9817850637522769,
                                     2: 0.507532956685499},
                       random_state=42)

In [43]:
y_pred = clf.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.82      0.84      0.83       122
           2       0.93      0.91      0.92       236

    accuracy                           0.88       360
   macro avg       0.58      0.58      0.58       360
weighted avg       0.89      0.88      0.88       360

Confusion Matrix:
 [[  0   2   0]
 [  3 103  16]
 [  1  21 214]]
